# dots.ocr on `platform-case-001` — Colab GPU runner

Companion to `ocr-benchmark/dots.ocr.benchmark.ipynb`, which cannot run locally:
dots.ocr is 3.04B parameters in BF16, roughly 9 GB resident, against a 16 GB
machine sitting at 1–2 GB free. A free Colab T4 has 16 GB of VRAM and fits it.

**The question this run answers: does dots.ocr read Sinhala?** Neither RapidOCR
nor docTR can. Both scored 0/2 on the NIC and both missed `extent` on the title
certificate. Roughly half the corpus is Sinhala, so recognition sits upstream of
every other finding in the benchmark.

Everything the local benchmark defines is inlined below — prompts, `smart_resize`,
`parse_dots`, `platform_norm` — copied rather than paraphrased so results stay
byte-comparable with runs already scored locally. Output is written in the exact
cache layout the local notebooks read, so you can drop it straight back in.

---

## Privacy — read before uploading

You are uploading **real client documents**: real names, NICs, addresses and
consideration amounts. `dots.ocr.benchmark.ipynb` cell 18 says not to. This
notebook exists because that call was made deliberately, so make it with the
facts in view:

- Colab runtimes are Google infrastructure. Assume uploaded files and cell
  outputs may persist beyond the session.
- **Do not save this notebook to Drive with outputs.** Outputs contain
  transcribed NICs and names.
- **Do not mount Drive.** Upload at runtime with the picker instead, so nothing
  is written to persistent storage.
- Run the cleanup cell at the end, then `Runtime → Disconnect and delete runtime`.
- Only `platform-case-001` goes up. Not the other three matters.

In [ ]:
# GPU check. Runtime -> Change runtime type -> T4 GPU if this reports no GPU.
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or "no GPU")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU."

In [ ]:
# pypdfium2 renders pages exactly as the local benchmark does. flash-attn is
# skipped: it takes ~20 min to build on Colab and eager attention is fine here.
%pip install -q transformers accelerate pypdfium2 huggingface_hub

## Which model id resolves

The local notebook uses `dots-studio/dots.ocr`. Model ids move between orgs, and
downloading 6 GB against a 404 wastes ten minutes, so the candidates are probed
first. If none resolve, search the Hub for `dots.ocr` and set `MODEL_ID` by hand.

In [ ]:
from huggingface_hub import model_info

CANDIDATES = ["dots-studio/dots.ocr", "rednote-hilab/dots.ocr", "dots-studio/dots.mocr"]
MODEL_ID = None
for candidate in CANDIDATES:
    try:
        model_info(candidate)
        MODEL_ID = candidate
        print(f"resolved: {candidate}")
        break
    except Exception as exc:
        print(f"  {candidate}: {type(exc).__name__}")
if MODEL_ID is None:
    raise SystemExit("No candidate resolved. Find the current id on huggingface.co "
                     "and set MODEL_ID manually.")

## Upload the case

Upload from `ocr-benchmark/cases/platform-case-001/`:

- the `source-*.pdf` files you want read — **`source-003` and `source-005` are the
  Sinhala ones**, so upload at least those two
- `expected-fields.json`, needed to score field recovery

Files land in the runtime's temp storage, not Drive. The cleanup cell deletes them.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

WORK = Path("/content/case")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

uploaded = files.upload()
for name, blob in uploaded.items():
    (WORK / name).write_bytes(blob)

pdfs = sorted(p for p in WORK.iterdir() if p.suffix.lower() == ".pdf")
print(f"\n{len(pdfs)} PDF(s):")
for p in pdfs:
    print(f"  {p.name}")
print("expected-fields.json:", (WORK / "expected-fields.json").is_file())

## The benchmark's definitions, inlined

Copied verbatim from `ocr-benchmark/`. Changing any of these makes the numbers
incomparable with the local runs:

| here | source |
| --- | --- |
| `PROMPTS` | `dots.ocr.benchmark.ipynb` cell 3, itself copied from `dots_ocr/utils/prompts.py` |
| `smart_resize` | `dots.ocr.benchmark.ipynb` cell 3 |
| `parse_dots` | `dots.ocr.benchmark.ipynb` cell 6 |
| `clamp_bbox` | `contract.py:41` |
| `platform_norm`, `value_in_text` | `normalize.py:36`, `normalize.py:134` |

In [ ]:
import json, math, re, time, unicodedata
from typing import Sequence

DPI = 200            # production raster_dpi; matches every local run
VARIANT = "original" # no preprocessing, so pixels match the docTR/RapidOCR runs
MATTER = "platform-case-001"
MAX_PAGES_PER_DOC = 2

LAYOUT_PROMPT = "\n".join([
    "Please output the layout information from the PDF image, including each layout element's bbox, its category, and the corresponding text content within the bbox.",
    "",
    "1. Bbox format: [x1, y1, x2, y2]",
    "",
    "2. Layout Categories: The possible categories are ['Caption', 'Footnote', 'Formula', 'List-item', 'Page-footer', 'Page-header', 'Picture', 'Section-header', 'Table', 'Text', 'Title'].",
    "",
    "3. Text Extraction & Formatting Rules:",
    "    - Picture: For the 'Picture' category, the text field should be omitted.",
    "    - Formula: Format its text as LaTeX.",
    "    - Table: Format its text as HTML.",
    "    - All Others (Text, Title, etc.): Format their text as Markdown.",
    "",
    "4. Constraints:",
    "    - The output text must be the original text from the image, with no translation.",
    "    - All layout elements must be sorted according to human reading order.",
    "",
    "5. Final Output: The entire output must be a single JSON object.",
    "",
])

PROMPTS = {
    "prompt_layout_all_en": LAYOUT_PROMPT,
    "prompt_ocr": "Extract the text content from this image.",
}

MIN_PIXELS, MAX_PIXELS, IMAGE_FACTOR = 3136, 11289600, 28


def smart_resize(width: int, height: int, factor: int = IMAGE_FACTOR,
                 min_pixels: int = MIN_PIXELS, max_pixels: int = MAX_PIXELS):
    # Qwen2-VL smart resize. The model returns pixel boxes in the RESIZED frame,
    # so getting this wrong misplaces every box while still looking plausible.
    h_bar = max(factor, round(height / factor) * factor)
    w_bar = max(factor, round(width / factor) * factor)
    if h_bar * w_bar > max_pixels:
        beta = math.sqrt((height * width) / max_pixels)
        h_bar = max(factor, math.floor(height / beta / factor) * factor)
        w_bar = max(factor, math.floor(width / beta / factor) * factor)
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h_bar = math.ceil(height * beta / factor) * factor
        w_bar = math.ceil(width * beta / factor) * factor
    return w_bar, h_bar


def clamp_bbox(bbox: Sequence[float]):
    # Order corners, clip to [0,1]. Top-left origin, y increases downward.
    x0, y0, x1, y1 = (float(v) for v in bbox)
    x0, x1 = min(x0, x1), max(x0, x1)
    y0, y1 = min(y0, y1), max(y0, y1)
    f = lambda v: min(1.0, max(0.0, v))
    return (f(x0), f(y0), f(x1), f(y1))


def parse_dots(raw: str, resized):
    # Their JSON output -> the benchmark's normalized bbox contract.
    text = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.M).strip()
    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\[.*\]|\{.*\}", text, re.S)
        if not match:
            return []
        try:
            payload = json.loads(match.group(0))
        except json.JSONDecodeError:
            return []
    items = payload if isinstance(payload, list) else (
        payload.get("layout") or payload.get("elements") or payload.get("results") or [])
    rw, rh = resized
    out = []
    for item in items:
        if not isinstance(item, dict):
            continue
        box = item.get("bbox") or item.get("box")
        if not box or len(box) != 4:
            continue
        x0, y0, x1, y1 = (float(v) for v in box)
        out.append({
            "bbox": list(clamp_bbox((x0 / rw, y0 / rh, x1 / rw, y1 / rh))),
            "category": str(item.get("category") or item.get("label") or "Text"),
            "text": str(item.get("text") or ""),
        })
    return out


def platform_norm(value: str) -> str:
    # Byte-identical to the platform's eval_extraction._norm.
    text = unicodedata.normalize("NFC", value)
    text = " ".join(text.split()).casefold()
    return text.replace(",", "").replace("/=", "").replace("/-", "").strip(" .")


def value_in_text(value: str, text: str) -> bool:
    needle = platform_norm(value or "")
    return bool(needle) and needle in platform_norm(text or "")


# Parser check on a synthetic response, so it is verified before any GPU time.
_demo = '```json\n[{"bbox":[140,280,700,340],"category":"Title","text":"CERTIFICATE"}]\n```'
_parsed = parse_dots(_demo, (1400, 2800))
assert _parsed and _parsed[0]["category"] == "Title"
assert abs(_parsed[0]["bbox"][0] - 0.1) < 1e-9, "bbox must normalize against the RESIZED frame"
print("parser check passed | smart_resize(1653, 2339) ->", smart_resize(1653, 2339))

In [ ]:
# Render at 200 DPI, no preprocessing — identical to render.render_doc(variant="original").
import pypdfium2 as pdfium
from PIL import Image

def render_doc(path, dpi=DPI, page_limit=MAX_PAGES_PER_DOC):
    pdf = pdfium.PdfDocument(path)
    try:
        count = len(pdf) if page_limit is None else min(len(pdf), page_limit)
        for index in range(count):
            yield index + 1, pdf[index].render(scale=dpi / 72.0).to_pil().convert("RGB")
    finally:
        pdf.close()

pages = []   # (doc_name, page_no, PIL image)
for pdf_path in pdfs:
    for page_no, image in render_doc(pdf_path):
        pages.append((pdf_path.name, page_no, image))
print(f"{len(pages)} pages rendered at {DPI} DPI")

## Load the model

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor

began = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, dtype=torch.bfloat16, device_map="cuda")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
print(f"loaded {MODEL_ID} in {time.time() - began:.0f}s")
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
def run_dots(image, mode="prompt_layout_all_en", max_tokens=8192):
    # Returns (raw_model_text, seconds). Resizing follows their smart_resize.
    target = smart_resize(*image.size)
    resized = image.resize(target, Image.LANCZOS)
    messages = [{"role": "user", "content": [
        {"type": "image", "image": resized},
        {"type": "text", "text": PROMPTS[mode]}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[resized], return_tensors="pt").to("cuda")
    began = time.time()
    generated = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    trimmed = generated[0][inputs["input_ids"].shape[1]:]
    return processor.decode(trimmed, skip_special_tokens=True), time.time() - began


RESULTS = Path("/content/dotsocr-cache")
RESULTS.mkdir(exist_ok=True)

def cache_path(doc_name, page_no):
    # The exact layout the local notebooks read: renders/engine-cache/dotsocr/...
    doc_id = f"{MATTER}/{doc_name}"
    safe = "".join(c if c.isalnum() or c in "-_." else "-" for c in doc_id)
    out = RESULTS / safe
    out.mkdir(parents=True, exist_ok=True)
    return out / f"p{page_no:04d}@{DPI}-{VARIANT}.json"


records = []
for doc_name, page_no, image in pages:
    dest = cache_path(doc_name, page_no)
    if dest.is_file():
        payload = json.loads(dest.read_text(encoding="utf-8"))
    else:
        try:
            raw, seconds = run_dots(image)
            elements = parse_dots(raw, smart_resize(*image.size))
            payload = {"elements": elements, "seconds": round(seconds, 2)}
            dest.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
        except Exception as exc:
            payload = {"elements": [], "seconds": None,
                       "error": f"{type(exc).__name__}: {exc}"[:200]}
            print(f"  ERROR {doc_name} p{page_no}: {payload['error']}")
    text = "\n".join(e["text"] for e in payload["elements"] if e["text"].strip())
    records.append({"document": doc_name, "page": page_no,
                    "elements": len(payload["elements"]),
                    "chars": len(re.sub(r"\s+", "", text)),
                    "seconds": payload.get("seconds"), "text": text})
    print(f"  {doc_name[:46]:48s} p{page_no}  {len(payload['elements']):3d} regions  "
          f"{payload.get('seconds')}s")

import pandas as pd
df = pd.DataFrame(records)
df[["document", "page", "elements", "chars", "seconds"]]

## Does it read Sinhala?

Character volume does not answer this — docTR emitted plenty of confident Latin
characters from Sinhala pages, which is worse than reading nothing because it
looks like a result. The test is whether the output is **in the Sinhala script**,
measured by Unicode block.

In [ ]:
def sinhala_ratio(text: str) -> float:
    # Share of letters that are Sinhala codepoints. The honest test.
    sinhala = len(re.findall(r"[඀-෿]", text or ""))
    latin = len(re.findall(r"[A-Za-z]", text or ""))
    total = sinhala + latin
    return sinhala / total if total else 0.0

df["sinhala"] = df["text"].fillna("").map(sinhala_ratio)
print(df[["document", "page", "chars", "sinhala"]].to_string(index=False))

hits = int((df["sinhala"] > 0.05).sum())
print(f"\npages with meaningful Sinhala output: {hits} of {len(df)}")
print("reference — RapidOCR and docTR both scored 0.0 on every page.")
if hits == 0:
    print("\ndots.ocr produced no Sinhala. Confirm a known-Sinhala page was in the")
    print("upload (source-003, source-005) before concluding the model cannot do it.")

## Critical-field recovery, same measure as the local runs

In [ ]:
expected_path = WORK / "expected-fields.json"
if not expected_path.is_file():
    print("expected-fields.json not uploaded — skipping field scoring")
else:
    expected = json.loads(expected_path.read_text(encoding="utf-8"))
    # schemas/critical-fields.json, inlined so the notebook needs no repo checkout.
    CRITICAL = {"titleCertificateNo", "cadastralMapNo", "blockNo", "sheetNo", "parcelNo",
                "extent", "extentSubjectToTransfer", "consideration", "transferorNic",
                "transfereeNic", "surveyPlanNo", "surveyorRegistration", "lotNo",
                "assessmentNumber", "notaryCode", "holderDateOfBirth"}

    doc_text = {name: "\n".join(g.sort_values("page")["text"].fillna(""))
                for name, g in df.groupby("document")}

    rows = []
    for doc_name, payload in expected.items():
        text = doc_text.get(doc_name)
        if text is None:
            continue
        for key, want in (payload.get("fields") or {}).items():
            rows.append({"document": doc_name.replace("source-", "")[:26], "field": key,
                         "critical": key in CRITICAL,
                         "found": value_in_text(str(want), text)})
    found = pd.DataFrame(rows)
    if found.empty:
        print("none of the uploaded documents have labelled fields")
    else:
        print("share of ground-truth values found verbatim (%):")
        print((found.groupby("critical")["found"].mean() * 100).round(1).to_string())
        print()
        print(found[["document", "field", "critical", "found"]].to_string(index=False))

## Layout categories — the other reason to care

The RaV-IDP entity router and the MADP parser both approximate layout from OCR
box geometry. dots.ocr returns categories and reading order directly, which is
strictly better input for both.

In [ ]:
from collections import Counter

tally = Counter()
for doc_name, page_no, _ in pages:
    dest = cache_path(doc_name, page_no)
    if dest.is_file():
        tally.update(e["category"] for e in
                     json.loads(dest.read_text(encoding="utf-8"))["elements"])
for name, count in tally.most_common():
    print(f"  {name:16s} {count}")
if not tally:
    print("no layout output — check the errors above")

## Detected layout on a page

**Renders a real client page in the notebook output.** Off by default. If you turn
it on, clear the output before saving or downloading this notebook.

In [ ]:
SHOW_CLIENT_PAGES = False

if not SHOW_CLIENT_PAGES:
    print("skipped — set SHOW_CLIENT_PAGES = True to draw detected layout")
else:
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle

    doc_name, page_no, image = pages[0]
    elements = json.loads(cache_path(doc_name, page_no).read_text(encoding="utf-8"))["elements"]
    palette = {c: f"C{i}" for i, c in enumerate(sorted({e["category"] for e in elements}))}
    fig, ax = plt.subplots(figsize=(10, 13))
    ax.imshow(image)
    w, h = image.size
    for element in elements:
        x0, y0, x1, y1 = clamp_bbox(element["bbox"])
        colour = palette[element["category"]]
        ax.add_patch(Rectangle((x0 * w, y0 * h), (x1 - x0) * w, (y1 - y0) * h,
                               fill=False, linewidth=2, edgecolor=colour))
        ax.text(x0 * w, y0 * h - 4, element["category"], fontsize=7, color=colour)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"dots.ocr layout — {doc_name} p{page_no}")
    plt.show()

## Bring the results home

Downloads a zip of the cached JSON — boxes, categories and transcribed text, no
page images. It still contains transcribed client values, so treat the zip as
client data.

Unpack it into `ocr-benchmark/renders/engine-cache/dotsocr/` (gitignored). The
directory and filename convention already match, so `dots.ocr.benchmark.ipynb`
and `make_overlays.py` pick the results up with no changes:

```powershell
uv run python ocr-benchmark/make_overlays.py --doc title-certificate --pages 1
```

In [ ]:
zip_path = shutil.make_archive("/content/dotsocr-results", "zip", RESULTS)
print(f"{Path(zip_path).stat().st_size / 1024:.0f} KB")
files.download(zip_path)

## Cleanup

Run this, then **Runtime → Disconnect and delete runtime**. Deleting files is not
a guarantee against Google-side retention — disconnecting the runtime is the part
that matters.

In [ ]:
for target in (WORK, RESULTS, Path("/content/dotsocr-results.zip")):
    if target.is_dir():
        shutil.rmtree(target)
    elif target.exists():
        target.unlink()
print("uploaded documents and rendered pages removed from this runtime")
print("now: Runtime -> Disconnect and delete runtime")